# TIỀN XỬ LÝ DỮ LIỆU VÀ KỸ THUẬT ĐẶC TRƯNG
## Đồ án môn học: Dự đoán nhu cầu xe đạp công cộng (UCI Bike Sharing Dataset)

Sổ tay này thực hiện các bước làm sạch dữ liệu, xử lý giá trị bất thường, loại bỏ các cột gây rò rỉ dữ liệu và đa cộng tuyến, chuẩn bị tập dữ liệu hoàn chỉnh để huấn luyện mô hình.


# 1. Khởi tạo môi trường và đọc dữ liệu


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Đọc tập dữ liệu theo giờ gốc
df = pd.read_csv('../data/hour.csv')
print(f"Kích thước tập dữ liệu gốc: {df.shape[0]} dòng, {df.shape[1]} cột")
df.head()

Kích thước tập dữ liệu gốc: 17379 dòng, 17 cột


# 2. Loại bỏ các cột định danh và chống rò rỉ dữ liệu (Data Leakage)

Dựa trên kết quả phân tích dữ liệu (EDA), nhóm em thực hiện loại bỏ:
- **`instant` và `dteday`:** Chỉ là số thứ tự và ngày ghi nhận dạng chuỗi, không mang tính quy luật để mô hình học nên nhóm em xóa đi.
- **`casual` và `registered`:** Vì tổng của hai cột này chính bằng biến mục tiêu ($cnt = casual + registered$). Nếu giữ lại, mô hình sẽ bị rò rỉ dữ liệu (Data Leakage), chỉ cần học phép cộng là đạt điểm tuyệt đối giả tạo mà không có giá trị dự đoán trong thực tế.


In [2]:
# Loại bỏ các cột định danh và các cột gây rò rỉ dữ liệu
drop_leakage_cols = ['instant', 'dteday', 'casual', 'registered']
df = df.drop(columns=drop_leakage_cols, errors='ignore')

print(f"Đã loại bỏ các cột: {drop_leakage_cols}")
print(f"Kích thước dữ liệu hiện tại: {df.shape}")

Đã loại bỏ các cột: ['instant', 'dteday', 'casual', 'registered']
Kích thước dữ liệu hiện tại: (17379, 13)


# 3. Loại bỏ đặc trưng đa cộng tuyến (atemp)

Ở bước khám phá dữ liệu, nhóm em thấy tương quan giữa `temp` (nhiệt độ thực tế) và `atemp` (nhiệt độ cảm nhận) lên tới xấp xỉ 0.99. Để tránh hiện tượng đa cộng tuyến làm sai lệch các hệ số hồi quy, nhóm em chỉ giữ lại biến đo lường khách quan là `temp` và xóa cột `atemp`.


In [3]:
# Loại bỏ cột atemp để triệt tiêu hiện tượng đa cộng tuyến
df = df.drop(columns=['atemp'], errors='ignore')

print("Đã loại bỏ cột 'atemp'.")
print(f"Kích thước dữ liệu hiện tại: {df.shape}")

Đã loại bỏ cột 'atemp'.
Kích thước dữ liệu hiện tại: (17379, 12)


# 4. Xử lý 22 dòng có độ ẩm bằng 0 (hum == 0)

Do cảm biến thời tiết bị trục trặc vào ngày 10/03/2011 nên có 22 dòng ghi nhận độ ẩm $0.00$. Để giữ trọn vẹn chuỗi thời gian liên tục 17,379 giờ, nhóm em thay thế các giá trị 0 này bằng giá trị **trung vị (Median)** của độ ẩm theo từng tháng và từng khung giờ tương ứng từ các ngày bình thường.


In [4]:
# Kiểm tra số lượng bản ghi hum == 0 trước khi xử lý
zero_hum_count = (df['hum'] == 0).sum()
print(f"Số dòng có độ ẩm hum == 0: {zero_hum_count}")

if zero_hum_count > 0:
    # Tính giá trị trung vị độ ẩm theo (tháng, giờ) cho các bản ghi hợp lệ
    valid_hum = df[df['hum'] > 0]
    median_hum_by_mnth_hr = valid_hum.groupby(['mnth', 'hr'])['hum'].median()

    # Thay thế các giá trị 0 bằng trung vị theo tháng và khung giờ tương ứng
    df['hum'] = df.apply(
        lambda row: median_hum_by_mnth_hr.get((row['mnth'], row['hr']), 0.59) if row['hum'] == 0 else row['hum'],
        axis=1
    )

print(f"Số dòng có độ ẩm hum == 0 sau khi xử lý: {(df['hum'] == 0).sum()}")
print(f"Giá trị độ ẩm nhỏ nhất hiện tại: {df['hum'].min():.4f}")

Số dòng có độ ẩm hum == 0 trước khi xử lý: 22
Số dòng có độ ẩm hum == 0 sau khi xử lý: 0
Giá trị độ ẩm nhỏ nhất hiện tại: 0.0800


# 5. Kiểm tra lại tập dữ liệu sau bước làm sạch cơ bản

Sau khi làm sạch, tập dữ liệu còn lại đúng 12 cột (11 đặc trưng độc lập và 1 biến mục tiêu `cnt`), không còn giá trị rỗng hay bất hợp lý, sẵn sàng cho các bước tiếp theo.


In [5]:
# Kiểm tra tổng quan kiểu dữ liệu và giá trị khuyết
print("=== THÔNG TIN DỮ LIỆU HIỆN TẠI ===")
df.info()
print("\n=== THỐNG KÊ MÔ TẢ CÁC BIẾN SỐ HỌC ===")
df[['temp', 'hum', 'windspeed', 'cnt']].describe().T

=== THÔNG TIN DỮ LIỆU HIỆN TẠI ===
<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   season      17379 non-null  int64  
 1   yr          17379 non-null  int64  
 2   mnth        17379 non-null  int64  
 3   hr          17379 non-null  int64  
 4   holiday     17379 non-null  int64  
 5   weekday     17379 non-null  int64  
 6   workingday  17379 non-null  int64  
 7   weathersit  17379 non-null  int64  
 8   temp        17379 non-null  float64
 9   hum         17379 non-null  float64
 10  windspeed   17379 non-null  float64
 11  cnt         17379 non-null  int64  
dtypes: float64(3), int64(9)
memory usage: 1.6 MB

=== THỐNG KÊ MÔ TẢ CÁC BIẾN SỐ HỌC ===


# 6. Xử lý ngoại lai bằng phương pháp IQR (Outlier Handling)

Ở bước khám phá dữ liệu, nhóm em đã xác định có 478 điểm ngoại lai của biến mục tiêu `cnt` theo tiêu chuẩn $Q_3 + 1.5 \times IQR$ theo từng mùa (chiếm 2.75% dữ liệu):
- **Bản chất ngoại lai:** Các điểm này phần lớn là giờ cao điểm tan tầm vào những ngày thời tiết đẹp. Tuy nhiên, các giá trị cực lớn (vọt lên gần 1.000 xe) sẽ tạo ra sai số bình phương khổng lồ $(y - \hat{y})^2$, khiến các mô hình như OLS Linear Regression và AdaBoost (vốn tự động tăng trọng số mẫu lỗi) bị hiện tượng học vẹt (overfitting).
- **Giải pháp xử lý:** Để vừa bảo vệ mô hình khỏi bị kéo lệch trọng số, vừa giữ trọn vẹn 17.379 giờ quan sát chuỗi thời gian liên tục, nhóm em áp dụng kỹ thuật **Capping (Winsorization)**: giới hạn trần các điểm vượt ngưỡng về đúng giá trị râu trên của từng mùa tương ứng (thay vì xóa bỏ làm mất dữ liệu đỉnh).
- **Đối với các biến đầu vào:** Nhóm biến thời gian và phân loại rời rạc có miền giá trị cố định nên không áp dụng ngoại lai thống kê; nhóm biến thời tiết (`temp`, `hum`, `windspeed`) đã được kiểm tra (xử lý xong 22 dòng lỗi cảm biến độ ẩm) và giữ nguyên biến thiên tự nhiên để chuẩn hóa ở bước sau.


In [ ]:
# Áp dụng Capping (Winsorization) theo ngưỡng IQR cho biến mục tiêu cnt
df_clean = df.copy()
# Chuyển cnt sang float để có thể gán giá trị ngưỡng râu trên dạng thập phân
df_clean['cnt'] = df_clean['cnt'].astype(float)

season_col = 'season' if 'season' in df_clean.columns else None
if season_col:
    for season in df_clean['season'].unique():
        sub_idx = df_clean[df_clean['season'] == season].index
        q1 = df_clean.loc[sub_idx, 'cnt'].quantile(0.25)
        q3 = df_clean.loc[sub_idx, 'cnt'].quantile(0.75)
        iqr = q3 - q1
        upper_limit = q3 + 1.5 * iqr
        mask = (df_clean['season'] == season) & (df_clean['cnt'] > upper_limit)
        df_clean.loc[mask, 'cnt'] = upper_limit
else:
    q1 = df_clean['cnt'].quantile(0.25)
    q3 = df_clean['cnt'].quantile(0.75)
    iqr = q3 - q1
    upper_limit = q3 + 1.5 * iqr
    df_clean['cnt'] = df_clean['cnt'].clip(upper=upper_limit)

print(f"Giá trị cnt lớn nhất trước khi xử lý ngoại lai: {df['cnt'].max():.1f}")
print(f"Giá trị cnt lớn nhất sau khi Capping IQR: {df_clean['cnt'].max():.1f}")
df = df_clean


# 7. Kỹ thuật tạo đặc trưng chu kỳ thời gian (Cyclical Feature Encoding)

Các biến thời gian như giờ trong ngày (`hr`: 0–23), tháng (`mnth`: 1–12), và ngày trong tuần (`weekday`: 0–6) có bản chất là chu kỳ tuần hoàn khép kín. Ví dụ: 23h đêm và 0h sáng thực tế chỉ cách nhau 1 giờ, nhưng nếu để dạng số nguyên tuyến tính thì các mô hình (như hồi quy tuyến tính, kNN) sẽ hiểu là chúng cách xa nhau 23 đơn vị.

Để mô hình hiểu được tính liên tục vòng tròn, nhóm em sử dụng hàm lượng giác $\sin$ và $\cos$ để ánh xạ các biến này lên đường tròn đơn vị:
$$x_{\sin} = \sin\left(\frac{2\pi \cdot t}{T}\right), \quad x_{\cos} = \cos\left(\frac{2\pi \cdot t}{T}\right)$$
với $T = 24$ (cho giờ), $T = 12$ (cho tháng), và $T = 7$ (cho thứ).

In [ ]:
# Mã hóa chu kỳ lượng giác sin/cos cho giờ (24), tháng (12) và thứ (7)
df['hr_sin'] = np.sin(2 * np.pi * df['hr'] / 24)
df['hr_cos'] = np.cos(2 * np.pi * df['hr'] / 24)

df['mnth_sin'] = np.sin(2 * np.pi * df['mnth'] / 12)
df['mnth_cos'] = np.cos(2 * np.pi * df['mnth'] / 12)

df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)

print("Đã tạo 6 đặc trưng chu kỳ lượng giác: hr_sin, hr_cos, mnth_sin, mnth_cos, weekday_sin, weekday_cos")
df[['hr', 'hr_sin', 'hr_cos', 'mnth', 'mnth_sin', 'mnth_cos']].head()

# 8. Tạo đặc trưng mới: Giờ cao điểm (is_rush_hour)

Dựa trên kết quả phân tích biểu đồ Fever chart ở bước EDA, các đỉnh nhu cầu cao nhất chỉ bùng nổ vào các ngày làm việc bình thường (`workingday == 1`) tại khung giờ sáng (7h, 8h, 9h) và chiều tối (17h, 18h, 19h). 

Nhóm em tạo biến nhị phân `is_rush_hour` để giúp các thuật toán nắm bắt trực tiếp tín hiệu giờ cao điểm.

In [ ]:
# Tạo đặc trưng nhị phân is_rush_hour
df['is_rush_hour'] = ((df['workingday'] == 1) & (df['hr'].isin([7, 8, 9, 17, 18, 19]))).astype(int)

rush_count = df['is_rush_hour'].sum()
print(f"Số giờ cao điểm (is_rush_hour = 1): {rush_count} / {len(df)} giờ (chiếm {rush_count/len(df)*100:.1f}%)")
print(f"Lượng xe thuê trung bình giờ cao điểm: {df[df['is_rush_hour'] == 1]['cnt'].mean():.1f} xe/giờ")
print(f"Lượng xe thuê trung bình giờ bình thường: {df[df['is_rush_hour'] == 0]['cnt'].mean():.1f} xe/giờ")

# 9. Mã hóa biến phân loại bằng One-Hot Encoding

Biến `season` (4 mùa) và `weathersit` (4 loại thời tiết) là các biến danh nghĩa (không có quan hệ thứ tự hơn kém định lượng). Để mô hình không bị hiểu nhầm giá trị $1, 2, 3, 4$ là cấp bậc tuyến tính, nhóm em áp dụng mã hóa One-Hot với tham số `drop_first=True` để tránh hiện tượng bẫy biến giả (Dummy Variable Trap).

In [ ]:
# Áp dụng One-Hot Encoding cho season và weathersit nếu chưa mã hóa
cat_cols = [c for c in ['season', 'weathersit'] if c in df.columns]
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

print(f"Kích thước dữ liệu hiện tại: {df.shape[0]} dòng, {df.shape[1]} cột")
print("Danh sách các thuộc tính:")
print(list(df.columns))

# 10. Đánh giá và Chọn lọc đặc trưng bằng Kiểm định Thống kê (Feature Selection)

Để đánh giá khách quan mức độ đóng góp của từng đặc trưng lên biến mục tiêu `cnt`, nhóm em áp dụng đồng thời hai phương pháp kiểm định thống kê:
1. **ANOVA F-test (`f_regression`):** Đo lường mối quan hệ tuyến tính giữa biến đầu vào và biến mục tiêu.
2. **Thông tin tương hỗ Mutual Information (`mutual_info_regression`):** Đo lường mức độ phụ thuộc phi tuyến dựa trên lý thuyết thông tin Shannon.

In [ ]:
from sklearn.feature_selection import f_regression, mutual_info_regression

# Tách tập đặc trưng X và biến mục tiêu y
X = df.drop(columns=['cnt'])
y = df['cnt']

# Tính toán F-Score và Mutual Information
f_scores, p_values = f_regression(X, y)
mi_scores = mutual_info_regression(X, y, random_state=42)

df_importance = pd.DataFrame({
    'Feature': X.columns,
    'F_Score': f_scores,
    'Mutual_Info': mi_scores
})

# Vẽ biểu đồ cột ngang so sánh 2 tiêu chuẩn
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
fig.suptitle('Xếp hạng Mức độ Ảnh hưởng của Đặc trưng lên Nhu cầu Thuê xe\n(ANOVA F-test vs. Mutual Information)', 
             fontsize=14, fontweight='bold', y=1.02)

# 1. Panel F-Score (Tuyến tính)
df_f = df_importance.sort_values('F_Score', ascending=True)
colors_f = ['#e74c3c' if f == 'is_rush_hour' else '#3498db' for f in df_f['Feature']]
axes[0].barh(df_f['Feature'], df_f['F_Score'], color=colors_f, edgecolor='black', alpha=0.85)
axes[0].set_title('1. Kiểm định ANOVA F-test (Quan hệ tuyến tính)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Giá trị F-Score', fontsize=11)
axes[0].grid(axis='x', linestyle='--', alpha=0.5)

# 2. Panel Mutual Information (Phi tuyến)
df_mi = df_importance.sort_values('Mutual_Info', ascending=True)
colors_mi = ['#e74c3c' if f == 'is_rush_hour' else '#2ecc71' for f in df_mi['Feature']]
axes[1].barh(df_mi['Feature'], df_mi['Mutual_Info'], color=colors_mi, edgecolor='black', alpha=0.85)
axes[1].set_title('2. Thông tin tương hỗ Mutual Information (Quan hệ phi tuyến)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hệ số Mutual Information (nats)', fontsize=11)
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../output/prep_feature_selection.png', dpi=300, bbox_inches='tight')
plt.show()

**Nhận xét từ kết quả kiểm định:**
- Cả hai phương pháp đều chỉ ra rằng **khung giờ trong ngày (`hr`, `hr_sin`, `hr_cos`)** và **nhiệt độ (`temp`)** là những đặc trưng có mức độ chi phối mạnh nhất đến nhu cầu thuê xe.
- Đặc biệt, đặc trưng mới do nhóm em tạo là **`is_rush_hour` đạt F-Score cao nhất vượt trội (hơn 6.300)**, chứng minh việc tạo biến giờ cao điểm dựa trên khám phá thực tế mang lại tín hiệu tuyến tính cực kỳ mạnh mẽ cho mô hình.
- Các đặc trưng chu kỳ $\sin/\cos$ cũng nằm trong nhóm có Mutual Information cao, cho thấy hiệu quả rõ rệt của kỹ thuật mã hóa chu kỳ lượng giác.